# Lekcija 18 (nastavak): Potvrde koje dokazuju da je *čovjek* odobrio radnju

Lekcija dokazuje što je **agent** učinio i što je **vrata** odlučila. Ova bilježnica dodaje nedostajuću polovicu: dokaz da je **imenovani čovjek** odobrio **točnu** radnju — zaseban, od čovjeka potpisan potpis preko cjelokupne kanonične radnje, verificiran offline.

Oba artefakta ovdje koriste **istu strukturu omotnice kao potvrde u lekciji**: ravni teret s poljem `type`, potpisan Ed25519 preko SHA-256 kanoničnih bajtova tereta, sa struktuiranim objektom `signature` dodatim (i izuzetim iz potpisanih bajtova). Potvrda o odobrenju je novi `type` (`human.approval.v1`) pored tipa radnje, tako da jedna `verify_chain` pokriva obje vrste artefakata istim kodnim putem koji ste izgradili u glavnoj bilježnici. Ovo je također oblik staze zasebnog potpisa u Internet-Draftu kojeg lekcija slijedi (draft-farley-acta-signed-receipts).

Jedno namjerno unapređenje u odnosu na demo verifier u glavnoj bilježnici: verifier ovdje rješava `signature.key_id` naspram **registrija priključenih ključeva** umjesto da vjeruje javnom ključu nošenom unutar potvrde. To je produkcijski stav koji preporučuje vlastiti kontrolni popis lekcije ("objavi javni ključ za verifikaciju"), i to je ono što čini krivotvorenje odbijanjem, a ne zaobilaženjem vlastitim ključem.

Pravilo koje ova bilježnica uči: **potpisano odobrenje samo po sebi nije autoritet.** Autoritet postoji samo ako potvrda o odobrenju i potvrda o radnji i dalje vežu istu kanoničnu radnju u trenutku izvršenja, pod verzijom politike, ključem i isticanjem koji su i dalje važeći, i odobrenjem koje još nije potrošeno. Svaki neuspjeh odbija s **jasnim razlogom**, tako da možete razlikovati *autoritet je istekao* od *izvršena radnja se promijenila*.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## Točna radnja

Jedinica odobrenja je **kanonički objekt radnje** — ne nejasna oznaka poput "odobriti povrat," već precizna, u potpunosti specificirana radnja. Potpisivanje čitavog objekta (i izvođenje sažetka iz njega) omogućuje nam da kasnije dokažemo da je čovjek odobrio *ovo* i ništa drugo.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## Jedna omotnica, dvije vlasti

Svaki zapis je omotnica lekcije: plosnati sadržaj s poljem `type`, plus objekt `signature` (`alg`, `sig`, `key_id`) koji **nije** dio potpisanih bajtova. `verify_envelope` je zajednički strukturni + provjera potpisa za oba tipa zapisa; kojim **registriranim ključem** se rješava `signature.key_id` ono je što drži vlasti odvojenima:

- **potvrda odobrenja** (`human.approval.v1`) — imenovani odobravatelj, puna kanonska akcija **i njen sažetak**, `policy_version`, vremenske oznake izdavanja i isteka. Jednokratna potrošnja se prati na razini lanca.
- **zapis o akciji** (`agent.action.v1`) — identitet agenta, `run_id`, isti kanonski sažetak akcije, ishod izvođenja + vremenska oznaka, i `parent_approval_ref`: `receipt_hash` odobrenja, ista konvencija kao `previous_receipt_hash` u lancu lekcije.

Zajedničko polje `action_digest` je veza na koju se veže. `key_id` živi u objektu potpisa samo kao pomoć pri pretraživanju: preusmjeravanje na drugi registrirani ključ uzrokuje pad provjere potpisa, stoga ne donosi ništa.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: gdje se zapravo odlučuje o povezivanju

`verify_chain` **nije** zgodan omotač preko dvije provjere potpisa. To je jedino mjesto gdje se zajednički kanonski `action_digest`, politika/ključ/isticanje **svježine** odobrenja, i **jednokratna upotreba** odobrenja zajedno provjeravaju, u odnosu na akciju koja se izvršava *upravo sada*.

Svaki neuspjeh odbija s **jasnim razlogom**, tako da čitatelj odbijanja može prepoznati je li autoritet zastario (politika se premjestila, ključ promijenjen, odobrenje isteklo, odobrenje potrošeno) ili je izvršena akcija promijenjena ispod još uvijek valjanog odobrenja (zamjena digest-a).


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## Što veza hvata

Svaki slučaj u nastavku neuspješno se zatvara **zatvoreno** s **različitim razlogom**. Prvi blok je klasični skup (manipulacija, zbunjeni zastupnik, ponavljanje, krivotvorina na bilo kojoj vlasti, nepravilni unos). Drugi blok je par koji svojstvo čini stvarnim umjesto tvrdnjom:

- **zastarjela vlast** — potpis je još uvijek valjan, ali se verzija politike pomaknula, ključ odobravatelja je uklonjen iz privezanog registra ili je odobrenje isteklo prije izvršenja;
- **zamjena sažetka** — valjano potpisani zapis o radnji čiji `parent_approval_ref` upućuje na *stvarno* odobrenje, ali kanonski sažetak radnje tog odobrenja ne odgovara radnji koja se zapravo izvršava.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## Što ovo dokazuje — a što ne dokazuje

**Dokazuje:** da je imenovani čovjek odobrio *ovu točno kanonsku radnju* (cjelokupna radnja + sažetak, potpisano ključem riješenim iz prikačenog registra), i da je agent izvršio *točno tu odobrenu radnju* (isti sažetak, potvrda vezana na odobrenje preko `receipt_hash`, vlastita konvencija lanca lekcije) — dok je verzija politike odobrenja, ključ i isteka još uvijek bila važeća, točno jednom. Ako bilo koja strana promijeni, lanac se zatvara, a razlog odbijanja vam govori **koje** svojstvo je prekršeno: zastarjela ovlast naspram promijenjene radnje.

**Ne dokazuje:** da je sučelje za odobrenje pokazalo čovjeku ono što je mislio da potpisuje (WYSIWYS je svoj problem), da ključ nije bio prisiljen ili ukraden prije rotacije, ili da su posljedice nizvodno bile u skladu s radnjom. Potpisano ≠ ovlašteno: valjani potpis nad zastarjelom politikom, rotiranim ključem, isteklim vremenom ili drugačijim sažetkom ovdje ne donosi ništa.

Dvije vrste potvrda dijele omot lekcije i jedan kod put provjere `verify_chain` namjerno: veza koju ste izgradili za potvrde radnji u glavnoj bilježnici je isti kod koji provjerava ljudsko odobrenje. Jedan ugovor za provjeru, odvojene prikačene ovlasti, povezane kanonskim sažetkom radnje i ničim više.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
